In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ── Apply all Task 2 cleaning steps ──
df = pd.read_csv('churnguard_data.csv')
df = df.drop('customerID', axis=1)
df = df.drop_duplicates()
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()

contract_map = {
    'Month-to-month': 'Month-to-month', 'month to month': 'Month-to-month',
    'month-to-month': 'Month-to-month', 'Monthly': 'Month-to-month',
    'One year': 'One year', 'One Year': 'One year',
    'one year': 'One year', '1 year': 'One year',
    'Two year': 'Two year', 'Two Year': 'Two year',
    'two year': 'Two year', '2 year': 'Two year',
}
df['Contract'] = df['Contract'].map(contract_map)

internet_map = {
    'DSL': 'DSL', 'dsl': 'DSL',
    'Fiber optic': 'Fiber optic', 'Fibre optic': 'Fiber optic',
    'FiberOptic': 'Fiber optic', 'fiber optic': 'Fiber optic',
    'No': 'No', 'None': 'No', 'none': 'No',
}
df['InternetService'] = df['InternetService'].map(internet_map)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df[df['tenure'] > 0]
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure'] = df['tenure'].fillna(int(df['tenure'].median()))

# ── Encode target ──
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── Encode categorical columns ──
cat_cols = ['gender', 'PhoneService', 'InternetService',
            'Contract', 'PaperlessBilling', 'PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# ── Separate X and y ──
X = df.drop('Churn', axis=1)
y = df['Churn']

# ── Train test split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# ── Train model ──
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# ── Evaluate ──
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stay', 'Churn']))

Accuracy: 0.6954

Classification Report:
              precision    recall  f1-score   support

        Stay       0.74      0.87      0.80       122
       Churn       0.48      0.29      0.36        52

    accuracy                           0.70       174
   macro avg       0.61      0.58      0.58       174
weighted avg       0.66      0.70      0.67       174



c:\Users\vemur\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Task 3 — Train Classification Model (ChurnGuard)

## Goal
Encode cleaned data, train Logistic Regression, evaluate churn prediction.

## Steps Applied

### 1. Encode Target Column
```python
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
```
Model needs numbers not text.

### 2. Encode Categorical Columns
```python
cat_cols = ['gender', 'PhoneService', 'InternetService',
            'Contract', 'PaperlessBilling', 'PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
```
get_dummies → converts categories to 0/1 columns
drop_first=True → avoids multicollinearity (dummy variable trap)

### 3. Train Test Split
```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
```
80% training, 20% testing.

### 4. Train Logistic Regression
```python
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
```

## Results
Accuracy: 0.6954 (69.54%)
Classification Report:
precision  recall  f1-score
Stay (0)         0.74    0.87      0.80
Churn (1)        0.48    0.29      0.36

## Analysis of Results

### What went well:
- Stay class → Recall=0.87 → model correctly identifies 87% of customers who stay ✅

### What went wrong:
- Churn class → Recall=0.29 → model only catches 29% of actual churners ❌
- This is the most important class for business!
- Missing 71% of churners = huge business loss

### Why did this happen?
Stay  → 122 samples
Churn →  52 samples
Class imbalance! Model is biased towards majority class (Stay).

### ConvergenceWarning — what does it mean?
lbfgs failed to converge → model needed more iterations to find best coefficients
Fix options:

Increase max_iter further (2000, 5000)
Scale the data using StandardScaler before training
(Logistic Regression converges faster with scaled data)


### How to improve this model:

Apply StandardScaler → helps convergence + improves accuracy
Use class_weight='balanced' → fixes class imbalance
Try Random Forest → handles imbalance better
Apply SMOTE → oversample minority class (Churn)
Lower threshold → increase Churn recall


## Key Learnings
- Always check class distribution before training
- High accuracy doesn't mean good model (accuracy paradox!)
- For churn detection → Recall on Churn class matters most
- Logistic Regression needs scaling for best performance
- ConvergenceWarning = model didn't fully learn → scale data or increase max_iter